In [1]:
import json
import os
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

sys.path.insert(0, "..")
from schema import PurchaseOrder

load_dotenv(find_dotenv(usecwd=True))
llm = OpenAI(base_url="http://localhost:4000", api_key=os.environ["LITELLM_MASTER_KEY"])
DATA = Path("../data")

In [2]:
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGVector

embeddings = OpenAIEmbeddings(
    model="o2c-embed",
    base_url="http://localhost:4000",
    api_key=os.environ["LITELLM_MASTER_KEY"],
    check_embedding_ctx_length=False,
)
db_url = (
    f"postgresql+psycopg://{os.environ['POSTGRES_USER']}:"
    f"{os.environ['POSTGRES_PASSWORD']}@localhost:5432/o2c"
)
store = PGVector(
    embeddings=embeddings,
    collection_name="contracts",
    connection=db_url,
    use_jsonb=True,
    pre_delete_collection=True,
)

Collection not found


In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
docs = []
for path in sorted((DATA / "contracts").glob("*.md")):
    customer_number = path.stem.split("-")[0]
    for chunk in splitter.split_text(path.read_text(encoding="utf-8")):
        docs.append(
            Document(
                page_content=chunk,
                metadata={"customer_number": customer_number, "source": path.name},
            )
        )

store.add_documents(docs)
print(
    f"Stored {len(docs)} chunks from {len(list((DATA / 'contracts').glob('*.md')))} files"
)

Stored 34 chunks from 10 files


In [4]:
hits = store.similarity_search(
    "agreed price for material 400103 Unterlegscheibe",
    k=3,
    filter={"customer_number": "1000107"},
)
for h in hits:
    print(h.metadata["source"], "|", h.page_content[:150].replace("\n", " "), "\n")

1000107.md | | Material No. | Description | Unit | Agreed unit price | |---|---|---|---:| | 400101 | Sechskantschraube DIN 933 M8x30 verzinkt, 100 Stk | PAK | 14.9 

1000107.md | | 400113 | Hydrauliköl HLP 46 | L | 3.45 | | 400114 | Industriereiniger Konzentrat, 10 l Kanister | PC | 38.90 | | 400115 | Gewindebohrer-Satz M3-M12  

1000107.md | ## 5. Price tolerance  Deviations of an order price from the price agreed in Annex A of up to 2% per line item are accepted without further approval.  



In [7]:
from decimal import Decimal

from pydantic import BaseModel

master = json.loads((DATA / "master_data.json").read_text(encoding="utf-8"))
customer_by_name = {c["name"]: c["customer_number"] for c in master["customers"]}
labels = json.loads((DATA / "labels.json").read_text(encoding="utf-8"))

In [8]:
class PriceTerms(BaseModel):
    agreed_unit_price: Decimal | None
    tolerance_pct: Decimal | None
    evidence: str


TERMS_PROMPT = f"""You read contract excerpts for Alpentech Industriebedarf GmbH.
Find (1) the agreed unit price for the given material and (2) the price tolerance in percent.
Use ONLY the excerpts. If a value is not in the excerpts, use null.
In "evidence", quote the table row or sentence you used.
Return only JSON matching: {json.dumps(PriceTerms.model_json_schema())}
"""

cache: dict = {}


def get_terms(
    customer_number: str, material_number: str | None, description: str
) -> PriceTerms:
    key = (customer_number, material_number or description)
    if key in cache:
        return cache[key]
    flt = {"customer_number": customer_number}
    hits = store.similarity_search(
        f"agreed unit price {material_number or ''} {description}", k=4, filter=flt
    )
    hits += store.similarity_search(
        "price tolerance deviation percent", k=1, filter=flt
    )
    context = "\n---\n".join(h.page_content for h in hits)
    response = llm.chat.completions.create(
        model="o2c-extract",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": TERMS_PROMPT},
            {
                "role": "user",
                "content": f"Material: {material_number} {description}\n\nExcerpts:\n{context}",
            },
        ],
        extra_body={
            "metadata": {"trace_name": "price-terms", "tags": ["poc", customer_number]}
        },
    )
    cache[key] = PriceTerms.model_validate_json(response.choices[0].message.content)
    return cache[key]

In [9]:
def check_po(po: PurchaseOrder) -> list[int]:
    customer_number = po.customer_number or customer_by_name[po.customer_name]
    flagged = []
    for line in po.lines:
        terms = get_terms(customer_number, line.material_number, line.description)
        if terms.agreed_unit_price is None:
            flagged.append(line.line_no)  # no agreed price found -> a human must check
            continue
        deviation = (
            abs(line.unit_price - terms.agreed_unit_price)
            / terms.agreed_unit_price
            * 100
        )
        if deviation > (terms.tolerance_pct or Decimal(0)):
            flagged.append(line.line_no)
    return flagged

In [10]:
correct, mistakes = 0, []
for gt in sorted((DATA / "ground_truth").glob("*.json")):
    doc_id = gt.stem
    po = PurchaseOrder.model_validate_json(gt.read_text(encoding="utf-8"))
    label = labels[doc_id]
    expected = [label["line_no"]] if label.get("price_deviation") else []
    got = check_po(po)
    if got == expected:
        correct += 1
    else:
        mistakes.append((doc_id, "expected", expected, "got", got))

print(f"Price check correct: {correct}/50")
for m in mistakes:
    print(m)

Price check correct: 50/50
